In [1]:
# python version
import sys
print(sys.executable)

C:\Users\felip\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
import sys
import os
import pandas as pd
import numpy as np
import torch
import time

sys.path.append(os.path.abspath('../src'))

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from PipelineBuilder import Pipeline
from CrossValidation import CrossValidation

In [3]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# pytorch threads
n_threads = os.cpu_count()
torch.set_num_threads(n_threads)
n_threads

8

In [5]:
## Loading data
data_loader = DataLoader(path_train="../data/train.csv", path_eval="../data/validation.csv", path_test="../data/test.csv")
train, validation, test = data_loader.load()
train.shape, validation.shape, test.shape

((29000, 2), (1000, 2), (10000, 2))

In [6]:
## Splitting data
data_splitter = DataSplitter(src_language='english', target_language='portuguese')
X_train, y_train = data_splitter.split(train)
X_val, y_val = data_splitter.split(validation)
X_test, y_test = data_splitter.split(test)

In [7]:
# DL Pipeline 
params = {'model_name': "LSTM", 'tokenizer_method': "full words", 'max_length': 40, 'vocab_size': 8000, 'embedding_dim': 128, 'hidden_dim': 128, 'num_layers': 1, 'dropout': 0.3, 
          'teacher_forcing': 0.5, 'max_length_decoded': 40, 'lr': 1e-3, 'epochs': 1, 'batch_size': 1024} 
params['device'] = "gpu"

pipeline = Pipeline(**params)
pipeline

,tokenizer_method,'full words'
,max_length,40
,vocab_size,8000
,embedding_dim,128
,hidden_dim,128
,num_layers,1
,dropout,0.3
,teacher_forcing,0.5
,max_length_decoded,40
,lr,0.001
,epochs,1


In [18]:
## Hyper-parameter tunning
cv = CrossValidation(pipeline=clone(pipeline))

param_grid = {
    'tokenizer_method': ["sentencepiece"],
    'max_length': [40],
    'vocab_size': [3000],
    'max_length_decoded': [40],
    'embedding_dim': [128], 
    'hidden_dim': [128], 
    'num_layers': [1],
    'teacher_forcing': [0.5], 
    'lr': [1e-3], 
    'epochs': [5], 
    'batch_size': [128]
}

cv_lstm_results, loss_curves, _ = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
                                                               reduced=True, train_size=10000, return_loss_curves=True)
cv_lstm_results

Time cleaning:  2.744640588760376
Time tokenizing:  4.121524810791016
Training Activated ? True
Loss after Epoch 1: 6.2644.
forward time: 250.4738096013898
backward time:  332.11215970129706
loss time:  5.807187000173144
Loss after gradient descent: 4.921322.
Total Training time: 600.460622549057.
Params:  {'batch_size': 128, 'embedding_dim': 128, 'epochs': 5, 'hidden_dim': 128, 'lr': 0.001, 'max_length': 40, 'max_length_decoded': 40, 'num_layers': 1, 'teacher_forcing': 0.5, 'tokenizer_method': 'sentencepiece', 'vocab_size': 3000}
Fit time: 607.3761103153229
time cleaning and encoding test:  1.7546014785766602
Training Activated ? False
time predicting test:  50.399537324905396
Time decodig test:  0.24996614456176758
Prediction time: 52.412230491638184

Total CV time: 662.7228882312775



,batch_size,embedding_dim,epochs,hidden_dim,lr,max_length,max_length_decoded,num_layers,teacher_forcing,tokenizer_method,vocab_size,fit_time,pred_time,final_loss,final_validation_score,score
0,128,128,5,128,0.001,40,40,1,0.5,sentencepiece,3000,607.37611,52.41223,4.921322,NaN,0.411867


In [ ]:
cv.pipeline.score(X_test, y_test)

In [20]:
preds = cv.pipeline.predict(X_test)
preds_df = pd.DataFrame({'english': X_test.iloc[:, 0].values, 'portuguese': y_test.iloc[:, 0], 'translation': preds})
preds_df.head(20)

time cleaning and encoding test:  1.8492650985717773
Training Activated ? False
time predicting test:  50.728140115737915
Time decodig test:  0.2557382583618164


,english,portuguese,translation
0,Everything matters.,Tudo importa.,que o..
1,I'm going to get something to eat.,Vou comer qualquer coisa.,Eu que o de.
2,Isn't there a pharmacy nearby?,Não há uma farmácia por aqui?,"que que que que não é a, não??"
3,The secret of longevity is to choose your pare...,Escolher os seus pais cuidadosamente é o segre...,A é um de de de de de de de de.
4,Tom isn't very likely to forget to do that.,Tom provavelmente não vai esquecer de fazer isso.,Tom não que que que que que que Tom.
5,They're both empty.,Ambos estão vazios.,que é a..
6,"We waited a long time, but Tom never came.",Nós esperamos por muito tempo mas o Tom nunca ...,Tom que que o que o que o que.
7,They held the meeting here.,Eles fizeram a reunião aqui.,Tom que o de de.
8,Are you ready to leave?,Você está pronto para partir?,que que que você?
9,I heard Tom partied all night.,Ouvi Tom festejando a noite toda.,Eu que que que que Tom que o Tom.
